# Portfolio: AI Data Analysis Agent (Local, Streamlit UI)

**Goal:** Build a reusable, portfolio-ready AI agent class that can analyze tabular data locally. This notebook demonstrates: uploading and preprocessing data, creating a reusable agent class (that wraps an LLM + SQL/Pandas tools), running natural-language queries, and producing reproducible results for your portfolio.

**Stack:** Python, pandas, duckdb, Streamlit (optional UI), OpenAI (or other LLM backend), and a small agent wrapper.

---
## 1 — Setup & Requirements

This notebook is designed to run locally. It includes an optional Streamlit-based UI so you can present the project as an interactive demo in your portfolio.

**Install required packages (run in your environment / virtualenv):**
```
pip install pandas duckdb streamlit openai tqdm python-dotenv
# pip install agno phi  # (if they exist in pip)
```

### Notes on LLM backends
You can connect any LLM that exposes a `chat`/`completion` interface. The example agent below uses a thin `LLMClient` wrapper so you can swap OpenAI, local LLMs, or mock implementations for testing.

In [ ]:
# Standard imports used throughout the notebook
import os
import json
import tempfile
import csv
from dataclasses import dataclass
from typing import Optional, Dict, Any, List
import pandas as pd
import duckdb
import textwrap
from tqdm import tqdm

# Optional: load environment variables from a .env file for API keys
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

## 2 — Utility: File preprocessing (safe local handling)
This cell provides a reusable function that accepts a local file path (CSV or XLSX), performs basic cleaning and type inference, and writes a temporary CSV file quoted for downstream SQL ingestion.

In [ ]:
def preprocess_and_save_local(file_path: str, na_values=None) -> (str, List[str], pd.DataFrame):
    """
    Read a CSV/XLSX file, coerce dates/numbers where possible, quote string fields, and save to a temporary CSV.
    Returns: (temp_path, columns_list, dataframe)
    """
    if na_values is None:
        na_values = ['NA', 'N/A', 'missing', '']

    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(file_path, encoding='utf-8', na_values=na_values)
    elif ext in ('.xls', '.xlsx'):
        df = pd.read_excel(file_path, na_values=na_values)
    else:
        raise ValueError('Unsupported file format: ' + ext)

    for col in df.select_dtypes(include=['object']):
        df[col] = df[col].astype(str).replace({r'"': '""'}, regex=True)

    for col in df.columns:
        try:
            if 'date' in col.lower():
                df[col] = pd.to_datetime(df[col], errors='coerce')
            elif df[col].dtype == 'object':
                df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            pass

    with tempfile.NamedTemporaryFile(delete=False, suffix='.csv') as tmpf:
        tmp_path = tmpf.name
    df.to_csv(tmp_path, index=False, quoting=csv.QUOTE_ALL)
    return tmp_path, df.columns.tolist(), df

## 3 — Reusable Agent Class (LLM client + DuckDB / Pandas tools)
The agent below is intentionally simple and modular:
- `LLMClient` is a thin wrapper you can replace (OpenAI, mock, etc.).
- `DataAnalysisAgent` manages a temporary DuckDB instance plus utilities to run SQL + Pandas operations.

In [ ]:
class LLMClient:
    def __init__(self, api_key: Optional[str] = None, model: str = 'gpt-4'):
        self.api_key = api_key or os.getenv('OPENAI_API_KEY')
        self.model = model

    def chat(self, messages: List[Dict[str, str]], temperature: float = 0.0) -> Dict[str, Any]:
        if not self.api_key:
            last_user = [m['content'] for m in messages if m['role'] == 'user'][-1]
            return {'content': f'MOCK RESPONSE — please configure OPENAI_API_KEY. You asked: {last_user}'}
        try:
            import openai
            openai.api_key = self.api_key
            resp = openai.ChatCompletion.create(model=self.model, messages=messages, temperature=temperature)
            content = resp['choices'][0]['message']['content']
            return {'content': content, 'raw': resp}
        except Exception as e:
            return {'content': f'LLM call failed: {e}'}

class DataAnalysisAgent:
    def __init__(self, llm: LLMClient, verbose: bool = False):
        self.llm = llm
        self.verbose = verbose
        self.conn = duckdb.connect(database=':memory:')
        self.registered_tables = {}

    def register_table_from_csv(self, csv_path: str, table_name: str = 'uploaded_data'):
        if self.verbose:
            print(f'Registering table {table_name} from {csv_path}')
        self.conn.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM read_csv_auto('{csv_path}')")
        self.registered_tables[table_name] = csv_path

    def sql(self, query: str) -> pd.DataFrame:
        if self.verbose:
            print('Running SQL:', query)
        df = self.conn.execute(query).df()
        return df

    def ask(self, natural_language_query: str, system_prompt: Optional[str] = None) -> Dict[str, Any]:
        system_prompt = system_prompt or 'You are an expert data analyst. Given a natural language question, produce a single SQL query that answers it. Return ONLY the SQL between ```sql and ``` markers.'
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': natural_language_query},
        ]
        llm_resp = self.llm.chat(messages)
        content = llm_resp.get('content', '')
        import re
        m = re.search(r'```sql\s*(.*?)```', content, re.DOTALL | re.IGNORECASE)
        if m:
            sql_query = m.group(1).strip()
        else:
            sql_query = content.strip()
        try:
            result_df = self.sql(sql_query)
            return {'sql': sql_query, 'result': result_df, 'llm_raw': llm_resp}
        except Exception as e:
            return {'error': str(e), 'llm_raw': llm_resp, 'sql_candidate': sql_query}

    def to_pandas(self, table_name: str) -> pd.DataFrame:
        return self.conn.execute(f'SELECT * FROM {table_name}').df()

    def close(self):
        try:
            self.conn.close()
        except Exception:
            pass

## 4 — Example: Wire it up locally + run queries
Replace `sample.csv` with your dataset.

In [ ]:
llm = LLMClient(api_key=os.getenv('OPENAI_API_KEY'))
agent = DataAnalysisAgent(llm=llm, verbose=True)

sample_df = pd.DataFrame({
    'user_id': [1,2,3,4,5],
    'signup_date': ['2020-01-01','2020-03-15','2020-03-20','2020-05-01','2020-07-12'],
    'country': ['US','US','GB','US','FR'],
    'revenue': [100.0, 150.25, 0.0, 300.5, 50.0]
})
tmp_csv = 'sample_demo.csv'
sample_df.to_csv(tmp_csv, index=False)
tmp_path, cols, df_loaded = preprocess_and_save_local(tmp_csv)
agent.register_table_from_csv(tmp_path, table_name='uploaded_data')

q = 'What is the total revenue by country, sorted by revenue descending?'
resp = agent.ask(q)
print('SQL executed:', resp.get('sql'))
display(resp.get('result'))